# BPR ベース + 2-hop 実験（劇的に上がりそうな実装を中心に）

**ベース**: 現在の最高スコア **0.76101**（`submission_atmacup_implicit_bpr16.csv`）と**同一**。  
`get_setup(use_best_pipeline=True)` で 55 特徴 + `get_bpr_base(ctx, factors=16)` で BPR 16 の 32 列 = 合計 87 特徴。  
「BPR のみ」の実験はこの提出と同じ特徴で学習するので、ベースがずれていない。  
ここに **factors 増加・内積 1 列・2-hop 集約・2-hop 割り算** を足してスコアの増減を確認する。

**協調フィルタで何を計算しているか**（行列→ベクトル→特徴の流れ）は `docs/08_COLLABORATIVE_FILTERING.md` の **§0 協調フィルタリングの計算の流れ（噛み砕き）** にまとめてある。

**実験の狙い（08 ドキュメントの「明日試すべき実装」より）**:

| カテゴリ | 実験 | 狙い |
|----------|------|------|
| factors 増 | BPR 32 / 64 | 現状 16 で 0.76101。次元を上げて表現力を増す。 |
| 内積 1 列 | 55 + bpr_dot のみ / BPR + bpr_dot | CF の「この批評家×この映画のスコア」を直接 LGB に渡す。 |
| 2-hop | Fresh 率平均・critic_te 平均・人数 | その映画をレビューした批評家たちの統計。 |
| 2-hop 割り算 | critic_te / movie_critic_te_mean | 1位で効いた「値/ユーザー平均」の映画版。 |

- ロジックは `lib/two_hop.py`。提出ファイルは `outputs/submissions/submission_2hop_{experiment_name}.csv`。

## 1. セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from lib.improvement_candidates import get_setup
from lib.two_hop import (
    run_experiment,
    run_experiment_dot_only,
    TWO_HOP_COLUMNS,
    TWO_HOP_FRESH_MEAN,
    TWO_HOP_CRITIC_TE_MEAN,
    TWO_HOP_REVIEW_COUNT,
    TWO_HOP_CRITIC_TE_DIV_MOVIE_TE,
)

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)

print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")
print(f"2-hop 列: {TWO_HOP_COLUMNS}")

Train: 653,507, Test: 40,716, Features: 55
提出先: outputs/submissions
2-hop 列: ['movie_fresh_rate_mean', 'movie_critic_te_mean', 'movie_review_count']


## 2. 劇的に上がりそうな実験（優先して実行）

BPR factors 増加・内積 1 列・2-hop 割り算を中心に並べている。提出して Public を確認し、効いたものを残して組み合わせを増やす。

In [2]:
def _report(name: str, r: dict):
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

results_high = []

# 0: BPR 16 のみ（ベースライン 0.76101 の再現）
r0 = run_experiment(ctx, "bpr_only", use_2hop_cols=None)
_report("0 BPR 16 のみ", r0)
results_high.append(r0)

# 1: BPR factors=32（次元増で表現力アップ）
r1 = run_experiment(ctx, "bpr32_only", use_2hop_cols=None, bpr_factors=32)
_report("1 BPR 32 のみ", r1)
results_high.append(r1)

# 2: BPR factors=64
r2 = run_experiment(ctx, "bpr64_only", use_2hop_cols=None, bpr_factors=64)
_report("2 BPR 64 のみ", r2)
results_high.append(r2)

# 3: 内積 1 列のみ（55 + bpr_dot）。CF のスコアを直接特徴に
r3 = run_experiment_dot_only(ctx, "bpr_dot_only")
_report("3 内積 1 列のみ", r3)
results_high.append(r3)

# 4: BPR 16 + 内積 1 列
r4 = run_experiment(ctx, "bpr_plus_dot", use_2hop_cols=None, use_bpr_dot=True)
_report("4 BPR 16 + 内積1列", r4)
results_high.append(r4)

# 5: BPR 16 + 全2-hop + 内積 1 列
r5 = run_experiment(ctx, "bpr_all2hop_plus_dot", use_2hop_cols=TWO_HOP_COLUMNS, use_bpr_dot=True)
_report("5 BPR + 全2-hop + 内積1列", r5)
results_high.append(r5)

# 6: 2-hop 割り算（critic_te / その映画の批評家平均 TE）。1位で効いた「値/ユーザー平均」の映画版
r6 = run_experiment(ctx, "bpr_2hop_ratio", use_2hop_cols=None, use_2hop_ratio=True)
_report("6 BPR + critic_te/movie_te_mean", r6)
results_high.append(r6)

# 7: BPR 32 + 2-hop 割り算
r7 = run_experiment(ctx, "bpr32_2hop_ratio", use_2hop_cols=None, bpr_factors=32, use_2hop_ratio=True)
_report("7 BPR 32 + 2hop_ratio", r7)
results_high.append(r7)

print(f"\n→ {sum(1 for r in results_high if r.get('ok'))} / {len(results_high)} 本 OK")

  0%|          | 0/100 [00:00<?, ?it/s]

  [0 BPR 16 のみ] → submission_2hop_bpr_only.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [1 BPR 32 のみ] → submission_2hop_bpr32_only.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

...

  [3 内積 1 列のみ] → submission_2hop_bpr_dot_only.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [4 BPR 16 + 内積1列] → submission_2hop_bpr_plus_dot.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

TypeError: Cannot setitem on a Categorical with a new category (0.6081423990879721), set the categories first

## 2.5 未提出だけ実行（まだ CSV が出ていない実験のみ回す）

§2 と §3 の全実験のうち、`submission_2hop_{name}.csv` が**まだない**ものだけを実行する。エラーで止まったあとや、追加した実験だけ回したいときにこのセルを使う。

In [2]:
# §2 と §3 の全実験の定義（name, 表示ラベル, use_2hop_cols, dot_only?, その他 kwargs）
ALL_EXPERIMENTS = [
    ("bpr_only", "0 BPR 16 のみ", None, False, {}),
    ("bpr32_only", "1 BPR 32 のみ", None, False, {"bpr_factors": 32}),
    ("bpr64_only", "2 BPR 64 のみ", None, False, {"bpr_factors": 64}),
    ("bpr_dot_only", "3 内積 1 列のみ", None, True, {}),
    ("bpr_plus_dot", "4 BPR 16 + 内積1列", None, False, {"use_bpr_dot": True}),
    ("bpr_all2hop_plus_dot", "5 BPR + 全2-hop + 内積1列", TWO_HOP_COLUMNS, False, {"use_bpr_dot": True}),
    ("bpr_2hop_ratio", "6 BPR + critic_te/movie_te_mean", None, False, {"use_2hop_ratio": True}),
    ("bpr32_2hop_ratio", "7 BPR 32 + 2hop_ratio", None, False, {"bpr_factors": 32, "use_2hop_ratio": True}),
    ("bpr_fresh", "BPR + movie_fresh_rate_mean", [TWO_HOP_FRESH_MEAN], False, {}),
    ("bpr_critic_te", "BPR + movie_critic_te_mean", [TWO_HOP_CRITIC_TE_MEAN], False, {}),
    ("bpr_count", "BPR + movie_review_count", [TWO_HOP_REVIEW_COUNT], False, {}),
    ("bpr_all2hop", "BPR + 全2-hop", TWO_HOP_COLUMNS, False, {}),
    ("bpr_all2hop_ratio", "BPR + 全2-hop + critic_te/movie_te", TWO_HOP_COLUMNS, False, {"use_2hop_ratio": True}),
]

def _report(name: str, r: dict):
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

todo = [e for e in ALL_EXPERIMENTS if not (ctx.submissions_dir / f"submission_2hop_{e[0]}.csv").exists()]
if not todo:
    print("未提出の実験はありません。")
else:
    print(f"未提出 {len(todo)} 本を実行します。\n")
    for name, label, use_2hop, dot_only, kw in todo:
        if dot_only:
            r = run_experiment_dot_only(ctx, name, bpr_factors=kw.get("bpr_factors", 16))
        else:
            r = run_experiment(ctx, name, use_2hop_cols=use_2hop, **kw)
        _report(label, r)
    print(f"\n→ 完了")

未提出 8 本を実行します。



  0%|          | 0/100 [00:00<?, ?it/s]

  [5 BPR + 全2-hop + 内積1列] → submission_2hop_bpr_all2hop_plus_dot.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [6 BPR + critic_te/movie_te_mean] → submission_2hop_bpr_2hop_ratio.csv  (OK)


...

  [BPR + movie_review_count] → submission_2hop_bpr_count.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR + 全2-hop] → submission_2hop_bpr_all2hop.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR + 全2-hop + critic_te/movie_te] → submission_2hop_bpr_all2hop_ratio.csv  (OK)

→ 完了


## 3. 2-hop バリエーション（1 列ずつ・全列）

その映画をレビューした批評家の統計を 1 列ずつ足して効く列を確認する。

In [ ]:
results_2hop = []

r1 = run_experiment(ctx, "bpr_fresh", use_2hop_cols=[TWO_HOP_FRESH_MEAN])
_report("BPR + movie_fresh_rate_mean", r1)
results_2hop.append(r1)

r2 = run_experiment(ctx, "bpr_critic_te", use_2hop_cols=[TWO_HOP_CRITIC_TE_MEAN])
_report("BPR + movie_critic_te_mean", r2)
results_2hop.append(r2)

r3 = run_experiment(ctx, "bpr_count", use_2hop_cols=[TWO_HOP_REVIEW_COUNT])
_report("BPR + movie_review_count", r3)
results_2hop.append(r3)

r4 = run_experiment(ctx, "bpr_all2hop", use_2hop_cols=TWO_HOP_COLUMNS)
_report("BPR + 全2-hop", r4)
results_2hop.append(r4)

# 2-hop 割り算 + 全2-hop の組み合わせ
r5 = run_experiment(ctx, "bpr_all2hop_ratio", use_2hop_cols=TWO_HOP_COLUMNS, use_2hop_ratio=True)
_report("BPR + 全2-hop + critic_te/movie_te", r5)
results_2hop.append(r5)

print(f"\n→ {sum(1 for r in results_2hop if r.get('ok'))} / {len(results_2hop)} 本 OK")

## 4. 1 本だけ試す（実験名・オプションを変えて実行）

1 本だけ `run_experiment` または `run_experiment_dot_only` を呼ぶ。`experiment_name` と `use_2hop_cols` / `use_bpr_dot` / `use_2hop_ratio` / `bpr_factors` を変えて追加実験する。

In [ ]:
# 例: BPR + fresh_mean と count の 2 列
# r = run_experiment(ctx, "bpr_fresh_count", use_2hop_cols=[TWO_HOP_FRESH_MEAN, TWO_HOP_REVIEW_COUNT])
# _report("BPR + fresh_count", r)

# 例: BPR 32 + 内積 + 2-hop 割り算
# r = run_experiment(ctx, "my_combo", bpr_factors=32, use_bpr_dot=True, use_2hop_ratio=True)
# _report("my_combo", r)

# 1 本だけ実行するときは下のコメントを外して編集
# r = run_experiment(ctx, "my_experiment", use_2hop_cols=[TWO_HOP_FRESH_MEAN], use_bpr_dot=True)
# _report("my_experiment", r)

## 5. 提出ファイル一覧

In [ ]:
for p in sorted(ctx.submissions_dir.glob("submission_2hop_*.csv")):
    print(p.name)